In [0]:
%sql
-- Reset Gold
-- TRUNCATE TABLE _exponent.omop_scm.provider;


In [ ]:
%sql
-- Reset Silver
-- DELETE FROM _exponent.omop_silver.provider
-- WHERE source_system = 'allscripts_scm';


In [ ]:
%sql
-- Reset Mapping
-- DELETE FROM _exponent.omop_mapping.source_to_provider
-- WHERE source_system = 'allscripts_scm';


In [0]:
%sql
-- Reset Gold
-- TRUNCATE TABLE _exponent.omop_scm.provider;


In [ ]:
%sql
-- Reset Silver
-- DELETE FROM _exponent.omop_silver.provider
-- WHERE source_system = 'allscripts_scm';


In [ ]:
%sql
-- Reset Mapping
-- DELETE FROM _exponent.omop_mapping.source_to_provider
-- WHERE source_system = 'allscripts_scm';


In [0]:
%sql
-- Create silver_provider temp view for SCM
CREATE OR REPLACE TEMP VIEW silver_provider AS
SELECT
    cp.DisplayName AS provider_name,
    npi.IDCode AS npi,
    dea.IDCode AS dea,
    0 AS specialty_concept_id,  -- TODO: Map Discipline to OMOP concept
    NULL AS care_site_id,
    NULL AS year_of_birth,  -- Not available in SCM
    0 AS gender_concept_id,  -- Not available in SCM
    CONCAT_WS(CHR(31), 'allscripts_scm', 'dbo_cv3careprovider', 'GUID', CAST(cp.GUID AS STRING)) AS provider_source_value,
    cp.Discipline AS specialty_source_value,
    0 AS specialty_source_concept_id,
    NULL AS gender_source_value,
    0 AS gender_source_concept_id,
    'allscripts_scm' AS source_system
FROM _exponent._bronze_allscripts_scm_prod01_vw.dbo_cv3careprovider cp
LEFT JOIN _exponent._bronze_allscripts_scm_prod01_vw.dbo_cv3careproviderid npi
    ON npi.ProviderGUID = cp.GUID
    AND npi.ProviderIDTypeCode = 'NPI'
    AND npi.Active = 1
LEFT JOIN _exponent._bronze_allscripts_scm_prod01_vw.dbo_cv3careproviderid dea
    ON dea.ProviderGUID = cp.GUID
    AND dea.ProviderIDTypeCode = 'DEA'
    AND dea.Active = 1
WHERE cp.Active = 1

In [0]:
%sql
MERGE INTO _exponent.omop_silver.provider AS target
USING silver_provider AS source
ON target.provider_source_value = source.provider_source_value

WHEN MATCHED AND NOT (
     target.provider_name               <=> source.provider_name
 AND target.npi                         <=> source.npi
 AND target.dea                         <=> source.dea
 AND target.specialty_concept_id        <=> source.specialty_concept_id
 AND target.care_site_id                <=> source.care_site_id
 AND target.year_of_birth               <=> source.year_of_birth
 AND target.gender_concept_id           <=> source.gender_concept_id
 AND target.specialty_source_value      <=> source.specialty_source_value
 AND target.specialty_source_concept_id <=> source.specialty_source_concept_id
 AND target.gender_source_value         <=> source.gender_source_value
 AND target.gender_source_concept_id    <=> source.gender_source_concept_id
 AND target.source_system               <=> source.source_system
)
THEN UPDATE SET
  target.provider_name               = source.provider_name,
  target.npi                         = source.npi,
  target.dea                         = source.dea,
  target.specialty_concept_id        = source.specialty_concept_id,
  target.care_site_id                = source.care_site_id,
  target.year_of_birth               = source.year_of_birth,
  target.gender_concept_id           = source.gender_concept_id,
  target.specialty_source_value      = source.specialty_source_value,
  target.specialty_source_concept_id = source.specialty_source_concept_id,
  target.gender_source_value         = source.gender_source_value,
  target.gender_source_concept_id    = source.gender_source_concept_id,
  target.source_system               = source.source_system,
  target.last_mod_tsp                = CURRENT_TIMESTAMP()

WHEN NOT MATCHED THEN INSERT (
  provider_name,
  npi,
  dea,
  specialty_concept_id,
  care_site_id,
  year_of_birth,
  gender_concept_id,
  provider_source_value,
  specialty_source_value,
  specialty_source_concept_id,
  gender_source_value,
  gender_source_concept_id,
  source_system,
  last_mod_tsp
) VALUES (
  source.provider_name,
  source.npi,
  source.dea,
  source.specialty_concept_id,
  source.care_site_id,
  source.year_of_birth,
  source.gender_concept_id,
  source.provider_source_value,
  source.specialty_source_value,
  source.specialty_source_concept_id,
  source.gender_source_value,
  source.gender_source_concept_id,
  source.source_system,
  CURRENT_TIMESTAMP()
);

In [0]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_provider (
    source_system,
    provider_source_value,
    active_flag,
    created_tsp,
    last_mod_tsp,
    merge_id,
    merge_reason
)
SELECT
    s.source_system,
    s.provider_source_value,
    TRUE AS active_flag,
    CURRENT_TIMESTAMP() AS created_tsp,
    COALESCE(s.last_mod_tsp, CURRENT_TIMESTAMP()) AS last_mod_tsp,
    NULL AS merge_id,
    NULL AS merge_reason
FROM (
    SELECT DISTINCT
        source_system,
        provider_source_value,
        last_mod_tsp
    FROM _exponent.omop_silver.provider
    WHERE provider_source_value IS NOT NULL
) s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_provider x
  ON s.provider_source_value = x.provider_source_value;

In [0]:
gold_df = spark.sql("""
SELECT
  x.provider_id,
  s.provider_name,
  s.npi,
  s.dea,
  s.specialty_concept_id,
  s.care_site_id,
  s.year_of_birth,
  s.gender_concept_id,
  s.provider_source_value,
  s.specialty_source_value,
  s.specialty_source_concept_id,
  s.gender_source_value,
  s.gender_source_concept_id,
  s.last_mod_tsp
FROM _exponent.omop_silver.provider s
JOIN _exponent.omop_mapping.source_to_provider x
  ON s.provider_source_value = x.provider_source_value
 AND x.active_flag = true
WHERE s.source_system = 'allscripts_scm'
""")
display(gold_df)
gold_df.createOrReplaceTempView("gold")

In [0]:
%sql
-- # SELECT provider_name, npi, dea, provider_source_value
-- # FROM _exponent.omop_silver.provider
-- # WHERE source_system = 'allscripts_scm'
-- # AND npi IS NOT NULL
-- # LIMIT 20

In [0]:
%sql
-- MERGE INTO _exponent.omop.provider AS target
MERGE INTO _exponent.omop_scm.provider AS target
USING gold AS source
ON target.provider_id = source.provider_id

WHEN MATCHED AND NOT (
     target.provider_name               <=> source.provider_name
 AND target.npi                         <=> source.npi
 AND target.dea                         <=> source.dea
 AND target.specialty_concept_id        <=> source.specialty_concept_id
 AND target.care_site_id                <=> source.care_site_id
 AND target.year_of_birth               <=> source.year_of_birth
 AND target.gender_concept_id           <=> source.gender_concept_id
 AND target.provider_source_value       <=> source.provider_source_value
 AND target.specialty_source_value      <=> source.specialty_source_value
 AND target.specialty_source_concept_id <=> source.specialty_source_concept_id
 AND target.gender_source_value         <=> source.gender_source_value
 AND target.gender_source_concept_id    <=> source.gender_source_concept_id
)
THEN UPDATE SET
  target.provider_name               = source.provider_name,
  target.npi                         = source.npi,
  target.dea                         = source.dea,
  target.specialty_concept_id        = source.specialty_concept_id,
  target.care_site_id                = source.care_site_id,
  target.year_of_birth               = source.year_of_birth,
  target.gender_concept_id           = source.gender_concept_id,
  target.provider_source_value       = source.provider_source_value,
  target.specialty_source_value      = source.specialty_source_value,
  target.specialty_source_concept_id = source.specialty_source_concept_id,
  target.gender_source_value         = source.gender_source_value,
  target.gender_source_concept_id    = source.gender_source_concept_id

WHEN NOT MATCHED THEN INSERT (
  provider_id,
  provider_name,
  npi,
  dea,
  specialty_concept_id,
  care_site_id,
  year_of_birth,
  gender_concept_id,
  provider_source_value,
  specialty_source_value,
  specialty_source_concept_id,
  gender_source_value,
  gender_source_concept_id
) VALUES (
  source.provider_id,
  source.provider_name,
  source.npi,
  source.dea,
  source.specialty_concept_id,
  source.care_site_id,
  source.year_of_birth,
  source.gender_concept_id,
  source.provider_source_value,
  source.specialty_source_value,
  source.specialty_source_concept_id,
  source.gender_source_value,
  source.gender_source_concept_id
);
